# CNN + OpenMax — Open-Set NIDS (Final Version)

**Network Intrusion Detection System** using CICIDS2017 dataset with open-set recognition.

This notebook trains a supervised CNN classifier on 6 known attack classes, then applies three
open-set detection methods to identify unknown (Infiltration) traffic:

1. **Standard OpenMax** — Weibull-calibrated logit recalibration
2. **Hybrid OpenMax** — penultimate-layer distances with logit-space recalibration (Bendale & Boult 2016)
3. **MSP Threshold** — Maximum Softmax Probability baseline

**Bugs fixed over previous versions:**
- `weibull_min` instead of `exponweib` (4-value unpack fix)
- All `model.predict()` results cast to `float32` to avoid mixed-precision overflow
- Class weights handle missing classes (Web Attack has 0 training samples)
- `classification_report` always receives explicit `labels` parameter
- OpenMax recalibration uses LOGITS (num_classes dim), not penultimate features (128 dim)
- Weibull distances computed in float64 to avoid inf from float16

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 1: Setup & Imports
# ═══════════════════════════════════════════════════════════════
import sys, os

HELPERS_DIR = "/kaggle/input/nids-helper-scripts"
sys.path.insert(0, HELPERS_DIR)

import infogan_model_kaggle as igo
import nids_utils_kaggle as nu
import openmax_kaggle_v2 as om  # FIXED version with weibull_min, hybrid_openmax_predict

# GPU + mixed precision
igo.setup_kaggle()

import numpy as np
import pandas as pd
import json, joblib, time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report, accuracy_score,
    precision_recall_fscore_support, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    multilabel_confusion_matrix,
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

print(f"TF version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2: Configuration
# ═══════════════════════════════════════════════════════════════
ALL_CSV_PATHS = [nu.MONDAY_CSV] + nu.ATTACK_FILES
SAVE_DIR = "/kaggle/working/cnn_openmax_final"
os.makedirs(SAVE_DIR, exist_ok=True)

KNOWN_CLASSES = [0, 1, 2, 3, 5, 6]  # Normal, Botnet, Brute Force, DoS, PortScan, Web Attack
UNKNOWN_CLASSES = [4]                 # Infiltration (held out)
NUM_KNOWN = len(KNOWN_CLASSES)        # 6

BATCH_SIZE = 512
EPOCHS = 50
LR = 0.001

TAIL_SIZE = 200
ALPHA_RANK = 5

known_class_names = [igo.CLASS_NAMES[c] for c in KNOWN_CLASSES]
print(f"Known classes ({NUM_KNOWN}): {known_class_names}")
print(f"Unknown: {[igo.CLASS_NAMES[c] for c in UNKNOWN_CLASSES]}")
print(f"\nAfter label remapping:")
for new_idx, old_idx in enumerate(KNOWN_CLASSES):
    print(f"  {igo.CLASS_NAMES[old_idx]} (original={old_idx}) -> remapped={new_idx}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3: Load/Fit Scaler
# ═══════════════════════════════════════════════════════════════
SCALER_PATH = os.path.join(nu.PREPROCESSING_DIR, "scaler_infogan_kaggle.pkl")
FEATURE_COLS_PATH = os.path.join(nu.PREPROCESSING_DIR, "infogan_kaggle_feature_cols.json")

if os.path.exists(SCALER_PATH) and os.path.exists(FEATURE_COLS_PATH):
    print("Loading existing scaler...")
    scaler = joblib.load(SCALER_PATH)
    with open(FEATURE_COLS_PATH) as f:
        feature_cols = json.load(f)
else:
    print("Fitting scaler from scratch...")
    scaler, feature_cols = igo.fit_scaler_streaming(ALL_CSV_PATHS, feature_range=(-1, 1))
    os.makedirs(os.path.dirname(SCALER_PATH), exist_ok=True)
    joblib.dump(scaler, SCALER_PATH)
    with open(FEATURE_COLS_PATH, "w") as f:
        json.dump(feature_cols, f)

print(f"Features: {len(feature_cols)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 4: Preprocess + Train/Val Split
# ═══════════════════════════════════════════════════════════════
import glob as _glob

# Clean old cached .npy files to avoid stale data
old_npy = _glob.glob(os.path.join(nu.PREPROCESSING_DIR, "*.npy"))
if old_npy:
    print(f"Cleaning {len(old_npy)} cached .npy files...")
    for f in old_npy:
        os.remove(f)

images_path, labels_path, total_samples = igo.preprocess_csvs_to_numpy(
    ALL_CSV_PATHS, scaler, feature_cols,
    known_classes=KNOWN_CLASSES, save_dir=nu.PREPROCESSING_DIR,
)

images = np.load(images_path)
labels = np.load(labels_path)
print(f"Total: {len(images):,} samples")
print(f"Class distribution:")
for i in range(NUM_KNOWN):
    count = np.sum(labels == i)
    print(f"  {known_class_names[i]} (class {i}): {count:,}")

X_train, X_val, y_train, y_val = train_test_split(
    images, labels, test_size=0.15, random_state=42, stratify=labels
)
del images, labels

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .cache().shuffle(50000)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print(f"\nTrain: {len(X_train):,}  Val: {len(X_val):,}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 5: Class Weights (FIXED — handles missing classes)
# ═══════════════════════════════════════════════════════════════
# BUG FIX: compute_class_weight("balanced", classes=np.arange(NUM_KNOWN), y=y_train)
# fails if a class has 0 samples (Web Attack / remapped class 5).
# Solution: compute only on classes present in y_train, then fill missing with 1.0.

unique_classes = np.unique(y_train)
cw_values = compute_class_weight("balanced", classes=unique_classes, y=y_train)

class_weight_dict = {}
for i in range(NUM_KNOWN):
    if i in unique_classes:
        idx = np.where(unique_classes == i)[0][0]
        class_weight_dict[i] = cw_values[idx]
    else:
        class_weight_dict[i] = 1.0  # missing class gets neutral weight

print("Class weights:")
for i, name in enumerate(known_class_names):
    present = "(present)" if i in unique_classes else "(MISSING — neutral weight)"
    print(f"  {name}: {class_weight_dict[i]:.4f} {present}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 6: Build CNN Classifier
# ═══════════════════════════════════════════════════════════════
def build_nids_classifier(num_classes):
    inp = layers.Input(shape=(11, 11, 1))

    x = layers.Conv2D(64, 4, strides=2, padding="same",
                      kernel_initializer="he_normal")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.2)(x)

    x = layers.Conv2D(128, 4, strides=2, padding="same",
                      kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.2)(x)

    x = layers.Conv2D(256, 4, strides=1, padding="same",
                      kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.2)(x)

    x = layers.Flatten()(x)

    # Penultimate layer — 128-dim (used for OpenMax distance computation)
    x = layers.Dense(128, kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.2, name="penultimate")(x)
    x = layers.Dropout(0.3)(x)

    # float32 output head — critical for mixed precision
    out = layers.Dense(num_classes, activation="softmax",
                       dtype="float32", name="class_output")(x)

    return keras.Model(inp, out, name="nids_classifier")

model = build_nids_classifier(NUM_KNOWN)
model.summary()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 7: Compile + Train
# ═══════════════════════════════════════════════════════════════
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        os.path.join(SAVE_DIR, "best_model.keras"),
        monitor="val_accuracy", save_best_only=True, mode="max", verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1,
    ),
]

print(f"Training for up to {EPOCHS} epochs (early stopping patience=10)...")
t0 = time.time()

history_obj = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1,
)

train_time = time.time() - t0
history = history_obj.history
print(f"\nTraining done in {train_time/60:.1f} min")
print(f"Best val_accuracy: {max(history['val_accuracy']):.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 8: Training Verification
# ═══════════════════════════════════════════════════════════════
final_val_acc = history["val_accuracy"][-1]
final_val_loss = history["val_loss"][-1]
best_val_acc = max(history["val_accuracy"])

print(f"Final val_accuracy: {final_val_acc:.4f}")
print(f"Best val_accuracy:  {best_val_acc:.4f}")
print(f"Final val_loss:     {final_val_loss:.4f}")

if best_val_acc < 0.5:
    print("\nWARNING: Val accuracy below 50% — model may not have learned properly.")
elif best_val_acc < 0.8:
    print("\nModel learned something but accuracy is moderate.")
else:
    print("\nModel training looks healthy.")

# Loss/accuracy plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")
axes[1].plot(history["accuracy"], label="train")
axes[1].plot(history["val_accuracy"], label="val")
axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("Epoch")
fig.suptitle("CNN Classifier Training Curves")
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "training_curves.png"), dpi=150)
plt.show()
print(f"Saved: {os.path.join(SAVE_DIR, 'training_curves.png')}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 9: Load Evaluation Data
# ═══════════════════════════════════════════════════════════════
# Known test set = validation split
X_test_known = X_val
y_test_known = y_val

# Unknown test set = Infiltration (class 4)
unk_img_path = os.path.join(nu.PREPROCESSING_DIR, "unknown_images.npy")
if os.path.exists(unk_img_path):
    X_test_unknown = np.load(unk_img_path)
else:
    print("Loading unknown (Infiltration) data from CSVs...")
    X_test_unknown, _ = igo.build_image_label_arrays_streamed(
        ALL_CSV_PATHS, scaler, feature_cols, known_classes=UNKNOWN_CLASSES
    )
    np.save(unk_img_path, X_test_unknown)

print(f"Train (for OpenMax fit): {len(X_train):,}")
print(f"Test (known classes):    {len(X_test_known):,}")
print(f"Test (unknown/Infiltr.): {len(X_test_unknown):,}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 10: Classifier Evaluation (no OpenMax)
# ═══════════════════════════════════════════════════════════════
# BUG FIX: .astype(np.float32) after model.predict() to avoid float16 overflow
# BUG FIX: labels=list(range(NUM_KNOWN)) to handle classes with 0 samples

y_pred_test = np.argmax(
    model.predict(X_test_known, batch_size=1024, verbose=0).astype(np.float32),
    axis=1
)

cls_acc = accuracy_score(y_test_known, y_pred_test)
print(f"CNN Classifier Test Accuracy: {cls_acc:.4f}")
print()
print(classification_report(
    y_test_known, y_pred_test,
    target_names=known_class_names,
    labels=list(range(NUM_KNOWN)),
    zero_division=0
))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 11: Extract Activations + Logits (CRITICAL CELL)
# ═══════════════════════════════════════════════════════════════
# BUG FIX: OpenMax openmax_predict() uses avs[:, :num_classes] for recalibration.
# Must pass LOGITS (num_classes dim), NOT penultimate features (128 dim).
#
# Strategy:
#   - Penultimate features (128-dim): for computing distances to MAVs (hybrid OpenMax)
#   - Logits (NUM_KNOWN-dim): for OpenMax score recalibration

# Extract penultimate layer (128-dim)
penultimate_model = keras.Model(model.input, model.get_layer("penultimate").output)

print("Extracting penultimate activations (train)...")
avs_train = penultimate_model.predict(X_train, batch_size=1024, verbose=0).astype(np.float32)

# Compute logits from penultimate features using the final Dense layer weights
final_dense = model.get_layer("class_output")
W_logit, b_logit = final_dense.get_weights()
logits_train = (avs_train @ W_logit + b_logit).astype(np.float64)

print(f"Penultimate shape: {avs_train.shape}")   # (N, 128)
print(f"Logits shape:      {logits_train.shape}") # (N, NUM_KNOWN)

# ── Test known ──
print("\nExtracting activations (test known)...")
avs_test_known = penultimate_model.predict(X_test_known, batch_size=1024, verbose=0).astype(np.float32)
logits_test_known = (avs_test_known @ W_logit + b_logit).astype(np.float64)

# ── Test unknown ──
print("Extracting activations (test unknown)...")
avs_test_unknown = penultimate_model.predict(X_test_unknown, batch_size=1024, verbose=0).astype(np.float32)
logits_test_unknown = (avs_test_unknown @ W_logit + b_logit).astype(np.float64)

# ── Softmax probabilities (for MSP method) ──
print("Computing softmax probabilities...")
softmax_known = model.predict(X_test_known, batch_size=1024, verbose=0).astype(np.float32)
softmax_unknown = model.predict(X_test_unknown, batch_size=1024, verbose=0).astype(np.float32)

print(f"\nAll activations extracted:")
print(f"  avs_train:         {avs_train.shape} (float32)")
print(f"  logits_train:      {logits_train.shape} (float64)")
print(f"  avs_test_known:    {avs_test_known.shape}")
print(f"  logits_test_known: {logits_test_known.shape}")
print(f"  avs_test_unknown:  {avs_test_unknown.shape}")
print(f"  logits_test_unknown: {logits_test_unknown.shape}")
print(f"  softmax_known:     {softmax_known.shape}")
print(f"  softmax_unknown:   {softmax_unknown.shape}")

## Method 1: Standard OpenMax (Logit-Space)

Compute MAVs and Weibull models directly on logit vectors (NUM_KNOWN dimensions).
Both distances and recalibration happen in logit space.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 13: Standard OpenMax on Logits
# ═══════════════════════════════════════════════════════════════

# MAVs and Weibull on logit space
mavs_logit, class_avs_logit = om.compute_mavs(logits_train, y_train, NUM_KNOWN)
distances_logit = om.compute_distances(class_avs_logit, mavs_logit, distance_type="euclidean")
weibull_logit = om.fit_weibull(distances_logit, tail_size=TAIL_SIZE)

# Print Weibull results
print("Method 1 — Weibull fits (logit space, euclidean):")
weibull_ok_m1 = 0
for cls, params in weibull_logit.items():
    name = known_class_names[cls]
    if params:
        status = f"c={params[0]:.4f}, scale={params[2]:.6f}"
        weibull_ok_m1 += 1
    else:
        status = "FAILED"
    print(f"  {name}: {status}")
print(f"  Weibull fit: {weibull_ok_m1}/{len(weibull_logit)} succeeded")

# Predict
print("\nPredicting on known test set...")
t0 = time.time()
preds_known_m1, probs_known_m1 = om.openmax_predict(
    logits_test_known, mavs_logit, weibull_logit,
    ALPHA_RANK, NUM_KNOWN, "euclidean"
)
print(f"  Done in {time.time()-t0:.1f}s")

print("Predicting on unknown test set...")
t0 = time.time()
preds_unknown_m1, probs_unknown_m1 = om.openmax_predict(
    logits_test_unknown, mavs_logit, weibull_logit,
    ALPHA_RANK, NUM_KNOWN, "euclidean"
)
print(f"  Done in {time.time()-t0:.1f}s")

# Metrics
known_acc_m1 = np.mean((preds_known_m1 != NUM_KNOWN) & (preds_known_m1 == y_test_known))
unknown_det_m1 = np.mean(preds_unknown_m1 == NUM_KNOWN)
h_score_m1 = 2 * known_acc_m1 * unknown_det_m1 / (known_acc_m1 + unknown_det_m1) if (known_acc_m1 + unknown_det_m1) > 0 else 0

print(f"\n--- Method 1: Standard OpenMax (Logit-Space, Euclidean) ---")
print(f"  Known Classification Accuracy: {known_acc_m1:.4f}")
print(f"  Unknown Detection Rate:        {unknown_det_m1:.4f}")
print(f"  H-Score:                       {h_score_m1:.4f}")

## Method 2: Hybrid OpenMax (Penultimate Distances + Logit Recalibration)

The proper Bendale & Boult (2016) formulation:
- Compute distances in **penultimate (feature) space** (128 dims) using cosine distance
- Fit Weibull on those feature-space distances
- Apply recalibration to **logit-space scores** (NUM_KNOWN dims)

This separates "how far from known clusters" (features) from "what are the class scores" (logits).

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 15: Hybrid OpenMax
# ═══════════════════════════════════════════════════════════════

# MAVs and Weibull on PENULTIMATE space (128-dim features)
mavs_pen, class_avs_pen = om.compute_mavs(avs_train, y_train, NUM_KNOWN)
distances_pen = om.compute_distances(class_avs_pen, mavs_pen, distance_type="cosine")
weibull_pen = om.fit_weibull(distances_pen, tail_size=TAIL_SIZE)

# Print Weibull results
print("Method 2 — Weibull fits (penultimate space, cosine):")
weibull_ok_m2 = 0
for cls, params in weibull_pen.items():
    name = known_class_names[cls]
    if params:
        status = f"c={params[0]:.4f}, scale={params[2]:.6f}"
        weibull_ok_m2 += 1
    else:
        status = "FAILED"
    print(f"  {name}: {status}")
print(f"  Weibull fit: {weibull_ok_m2}/{len(weibull_pen)} succeeded")

# Hybrid predict: penultimate distances + logit recalibration
print("\nPredicting on known test set (hybrid)...")
t0 = time.time()
preds_known_m2, probs_known_m2 = om.hybrid_openmax_predict(
    logits_test_known, avs_test_known, mavs_pen, weibull_pen,
    ALPHA_RANK, NUM_KNOWN, "cosine"
)
print(f"  Done in {time.time()-t0:.1f}s")

print("Predicting on unknown test set (hybrid)...")
t0 = time.time()
preds_unknown_m2, probs_unknown_m2 = om.hybrid_openmax_predict(
    logits_test_unknown, avs_test_unknown, mavs_pen, weibull_pen,
    ALPHA_RANK, NUM_KNOWN, "cosine"
)
print(f"  Done in {time.time()-t0:.1f}s")

# Metrics
known_acc_m2 = np.mean((preds_known_m2 != NUM_KNOWN) & (preds_known_m2 == y_test_known))
unknown_det_m2 = np.mean(preds_unknown_m2 == NUM_KNOWN)
h_score_m2 = 2 * known_acc_m2 * unknown_det_m2 / (known_acc_m2 + unknown_det_m2) if (known_acc_m2 + unknown_det_m2) > 0 else 0

print(f"\n--- Method 2: Hybrid OpenMax (Penultimate + Logit, Cosine) ---")
print(f"  Known Classification Accuracy: {known_acc_m2:.4f}")
print(f"  Unknown Detection Rate:        {unknown_det_m2:.4f}")
print(f"  H-Score:                       {h_score_m2:.4f}")

## Method 3: Maximum Softmax Probability (MSP) Threshold

Simple baseline: if the model's maximum softmax confidence is below a threshold,
classify the sample as unknown. Often surprisingly effective.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 17: MSP Threshold
# ═══════════════════════════════════════════════════════════════

# Confidence statistics
max_conf_known = softmax_known.max(axis=1)
max_conf_unknown = softmax_unknown.max(axis=1)
print(f"Known confidence:   min={max_conf_known.min():.4f}, mean={max_conf_known.mean():.4f}, median={np.median(max_conf_known):.4f}")
print(f"Unknown confidence: min={max_conf_unknown.min():.4f}, mean={max_conf_unknown.mean():.4f}, median={np.median(max_conf_unknown):.4f}")

# Sweep thresholds
sweep_results = om.sweep_msp_thresholds(softmax_known, y_test_known, softmax_unknown, NUM_KNOWN)
sweep_df = pd.DataFrame(sweep_results).sort_values("h_score", ascending=False)
print("\nTop 10 thresholds by H-Score:")
print(sweep_df.head(10).to_string(index=False))

best_th = sweep_df.iloc[0]["threshold"]
print(f"\nBest MSP threshold: {best_th}")

# Final MSP prediction with best threshold
preds_known_m3, probs_known_m3 = om.softmax_threshold_predict(softmax_known, best_th, NUM_KNOWN)
preds_unknown_m3, probs_unknown_m3 = om.softmax_threshold_predict(softmax_unknown, best_th, NUM_KNOWN)

# Metrics
known_acc_m3 = np.mean((preds_known_m3 != NUM_KNOWN) & (preds_known_m3 == y_test_known))
unknown_det_m3 = np.mean(preds_unknown_m3 == NUM_KNOWN)
h_score_m3 = 2 * known_acc_m3 * unknown_det_m3 / (known_acc_m3 + unknown_det_m3) if (known_acc_m3 + unknown_det_m3) > 0 else 0

print(f"\n--- Method 3: MSP Threshold (threshold={best_th}) ---")
print(f"  Known Classification Accuracy: {known_acc_m3:.4f}")
print(f"  Unknown Detection Rate:        {unknown_det_m3:.4f}")
print(f"  H-Score:                       {h_score_m3:.4f}")

## Comprehensive Comparison

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 19: Side-by-Side Comparison Table
# ═══════════════════════════════════════════════════════════════

def compute_method_metrics(preds_known, preds_unknown, y_true_known, num_known):
    """Compute all open-set metrics for one method."""
    known_acc = np.mean((preds_known != num_known) & (preds_known == y_true_known))
    known_rejection = np.mean(preds_known == num_known)
    unknown_det = np.mean(preds_unknown == num_known)
    false_known = 1.0 - unknown_det
    h_score = 2 * known_acc * unknown_det / (known_acc + unknown_det) if (known_acc + unknown_det) > 0 else 0
    
    # Overall accuracy (known correct + unknown detected) / total
    total = len(preds_known) + len(preds_unknown)
    correct = np.sum((preds_known != num_known) & (preds_known == y_true_known)) + np.sum(preds_unknown == num_known)
    overall_acc = correct / total if total > 0 else 0
    
    return {
        "known_acc": known_acc,
        "known_rejection": known_rejection,
        "unknown_det": unknown_det,
        "false_known": false_known,
        "h_score": h_score,
        "overall_accuracy": overall_acc,
    }

m1_metrics = compute_method_metrics(preds_known_m1, preds_unknown_m1, y_test_known, NUM_KNOWN)
m2_metrics = compute_method_metrics(preds_known_m2, preds_unknown_m2, y_test_known, NUM_KNOWN)
m3_metrics = compute_method_metrics(preds_known_m3, preds_unknown_m3, y_test_known, NUM_KNOWN)

comparison_data = {
    "Metric": ["Known Classification Acc", "Known Rejection Rate",
               "Unknown Detection Rate", "False Known Rate",
               "H-Score (KCA x UDR)", "Overall Accuracy"],
    "M1: Std OpenMax (Logit)": [
        m1_metrics["known_acc"], m1_metrics["known_rejection"],
        m1_metrics["unknown_det"], m1_metrics["false_known"],
        m1_metrics["h_score"], m1_metrics["overall_accuracy"]
    ],
    "M2: Hybrid OpenMax": [
        m2_metrics["known_acc"], m2_metrics["known_rejection"],
        m2_metrics["unknown_det"], m2_metrics["false_known"],
        m2_metrics["h_score"], m2_metrics["overall_accuracy"]
    ],
    "M3: MSP Threshold": [
        m3_metrics["known_acc"], m3_metrics["known_rejection"],
        m3_metrics["unknown_det"], m3_metrics["false_known"],
        m3_metrics["h_score"], m3_metrics["overall_accuracy"]
    ],
}

comparison_df = pd.DataFrame(comparison_data)
print("=" * 90)
print("  COMPREHENSIVE COMPARISON — ALL 3 OPEN-SET METHODS")
print("=" * 90)
print(comparison_df.to_string(index=False, float_format="%.4f"))

# Per-class precision/recall/F1 for each method
print("\n" + "=" * 90)
print("  PER-CLASS CLASSIFICATION REPORTS")
print("=" * 90)

all_class_names = known_class_names + ["Unknown"]
num_all = NUM_KNOWN + 1

for method_name, pk, pu in [
    ("Method 1: Standard OpenMax", preds_known_m1, preds_unknown_m1),
    ("Method 2: Hybrid OpenMax", preds_known_m2, preds_unknown_m2),
    ("Method 3: MSP Threshold", preds_known_m3, preds_unknown_m3),
]:
    y_true_all = np.concatenate([y_test_known, np.full(len(preds_unknown_m1), NUM_KNOWN)])
    y_pred_all = np.concatenate([pk, pu])
    print(f"\n--- {method_name} ---")
    print(classification_report(
        y_true_all, y_pred_all,
        target_names=all_class_names,
        labels=list(range(num_all)),
        zero_division=0
    ))

# Save comparison
comparison_df.to_csv(os.path.join(SAVE_DIR, "comparison_summary.csv"), index=False)
print(f"\nSaved: {os.path.join(SAVE_DIR, 'comparison_summary.csv')}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 20: Hyperparameter Sweep for All Methods
# ═══════════════════════════════════════════════════════════════

all_sweep_results = []

# ── Method 1: Standard OpenMax sweep ──
print("Sweeping Method 1 (Standard OpenMax on logits)...")
for ts in [20, 50, 100, 200, 500]:
    wp = om.fit_weibull(distances_logit, tail_size=ts)
    for ar in [1, 2, 3, 5, 7, 10]:
        pk, _ = om.openmax_predict(logits_test_known, mavs_logit, wp, ar, NUM_KNOWN, "euclidean")
        pu, _ = om.openmax_predict(logits_test_unknown, mavs_logit, wp, ar, NUM_KNOWN, "euclidean")
        ka = np.mean((pk != NUM_KNOWN) & (pk == y_test_known))
        ud = np.mean(pu == NUM_KNOWN)
        hs = 2 * ka * ud / (ka + ud) if (ka + ud) > 0 else 0
        all_sweep_results.append({
            "method": "M1_StdOpenMax_Logit",
            "tail_size": ts, "alpha_rank": ar, "threshold": None,
            "known_acc": round(ka, 4), "unknown_det": round(ud, 4),
            "h_score": round(hs, 4),
        })

# ── Method 2: Hybrid OpenMax sweep ──
print("Sweeping Method 2 (Hybrid OpenMax)...")
for ts in [20, 50, 100, 200, 500]:
    wp = om.fit_weibull(distances_pen, tail_size=ts)
    for ar in [1, 2, 3, 5, 7, 10]:
        pk, _ = om.hybrid_openmax_predict(
            logits_test_known, avs_test_known, mavs_pen, wp, ar, NUM_KNOWN, "cosine"
        )
        pu, _ = om.hybrid_openmax_predict(
            logits_test_unknown, avs_test_unknown, mavs_pen, wp, ar, NUM_KNOWN, "cosine"
        )
        ka = np.mean((pk != NUM_KNOWN) & (pk == y_test_known))
        ud = np.mean(pu == NUM_KNOWN)
        hs = 2 * ka * ud / (ka + ud) if (ka + ud) > 0 else 0
        all_sweep_results.append({
            "method": "M2_HybridOpenMax",
            "tail_size": ts, "alpha_rank": ar, "threshold": None,
            "known_acc": round(ka, 4), "unknown_det": round(ud, 4),
            "h_score": round(hs, 4),
        })

# ── Method 3: MSP Threshold sweep ──
print("Sweeping Method 3 (MSP Threshold)...")
msp_sweep = om.sweep_msp_thresholds(softmax_known, y_test_known, softmax_unknown, NUM_KNOWN)
for row in msp_sweep:
    all_sweep_results.append({
        "method": "M3_MSP_Threshold",
        "tail_size": None, "alpha_rank": None,
        "threshold": row["threshold"],
        "known_acc": row["known_acc"],
        "unknown_det": row["unknown_det"],
        "h_score": row["h_score"],
    })

sweep_all_df = pd.DataFrame(all_sweep_results).sort_values("h_score", ascending=False)
sweep_all_df.to_csv(os.path.join(SAVE_DIR, "hyperparameter_sweep_all.csv"), index=False)

print("\n" + "=" * 90)
print("  TOP 15 CONFIGURATIONS ACROSS ALL METHODS (by H-Score)")
print("=" * 90)
print(sweep_all_df.head(15).to_string(index=False))

best_config = sweep_all_df.iloc[0]
print(f"\nBest overall config:")
print(f"  Method:      {best_config['method']}")
print(f"  H-Score:     {best_config['h_score']:.4f}")
print(f"  Known Acc:   {best_config['known_acc']:.4f}")
print(f"  Unknown Det: {best_config['unknown_det']:.4f}")
if pd.notna(best_config['tail_size']):
    print(f"  Tail Size:   {int(best_config['tail_size'])}")
    print(f"  Alpha Rank:  {int(best_config['alpha_rank'])}")
if pd.notna(best_config['threshold']):
    print(f"  Threshold:   {best_config['threshold']}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 21: Detailed Metrics for Best Method
# ═══════════════════════════════════════════════════════════════

# Determine best method based on H-score from default config
method_scores = {
    "M1: Standard OpenMax (Logit)": (h_score_m1, preds_known_m1, preds_unknown_m1, probs_known_m1, probs_unknown_m1),
    "M2: Hybrid OpenMax": (h_score_m2, preds_known_m2, preds_unknown_m2, probs_known_m2, probs_unknown_m2),
    "M3: MSP Threshold": (h_score_m3, preds_known_m3, preds_unknown_m3, probs_known_m3, probs_unknown_m3),
}

best_method_name = max(method_scores, key=lambda k: method_scores[k][0])
best_hs, best_pk, best_pu, best_probs_k, best_probs_u = method_scores[best_method_name]

print(f"Best method by H-Score: {best_method_name} (H={best_hs:.4f})")
print("=" * 80)

# Combined true/pred
y_true_all = np.concatenate([y_test_known, np.full(len(X_test_unknown), NUM_KNOWN)])
y_pred_all = np.concatenate([best_pk, best_pu])
probs_all = np.concatenate([best_probs_k, best_probs_u])

all_class_names = known_class_names + ["Unknown"]
num_all = NUM_KNOWN + 1

# Per-class metrics
prec, rec, f1, support = precision_recall_fscore_support(
    y_true_all, y_pred_all, labels=list(range(num_all)), zero_division=0
)
mcm = multilabel_confusion_matrix(y_true_all, y_pred_all, labels=list(range(num_all)))
fpr_list = []
for i in range(num_all):
    tn, fp, fn, tp = mcm[i].ravel()
    fpr_list.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)

metrics_df = pd.DataFrame({
    "Class": all_class_names,
    "Precision": prec,
    "Recall": rec,
    "F1-Score": f1,
    "FPR": fpr_list,
    "Support": support.astype(int),
})

# Add macro and weighted averages
metrics_df = pd.concat([metrics_df, pd.DataFrame([{
    "Class": "Macro Avg",
    "Precision": prec.mean(),
    "Recall": rec.mean(),
    "F1-Score": f1.mean(),
    "FPR": np.mean(fpr_list),
    "Support": support.sum(),
}, {
    "Class": "Weighted Avg",
    "Precision": np.average(prec, weights=np.maximum(support, 1)),
    "Recall": np.average(rec, weights=np.maximum(support, 1)),
    "F1-Score": np.average(f1, weights=np.maximum(support, 1)),
    "FPR": np.average(fpr_list, weights=np.maximum(support, 1)),
    "Support": support.sum(),
}])], ignore_index=True)

print(f"\nDetailed Metrics — {best_method_name}")
print("-" * 80)
print(metrics_df.to_string(index=False, float_format="%.4f"))
print(f"\nOverall Accuracy: {accuracy_score(y_true_all, y_pred_all):.4f}")

metrics_df.to_csv(os.path.join(SAVE_DIR, "performance_metrics.csv"), index=False)
print(f"Saved: {os.path.join(SAVE_DIR, 'performance_metrics.csv')}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 22: ROC + PR Curves
# ═══════════════════════════════════════════════════════════════

y_true_bin = label_binarize(y_true_all, classes=list(range(num_all)))
if y_true_bin.shape[1] == 1:
    y_true_bin = np.hstack([1 - y_true_bin, y_true_bin])

colors = plt.cm.tab10(np.linspace(0, 1, num_all))

# ── ROC Curves ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
auc_scores = {}

# Per-class ROC
ax = axes[0]
for i in range(num_all):
    if y_true_bin[:, i].sum() == 0:
        continue
    fpr_i, tpr_i, _ = roc_curve(y_true_bin[:, i], probs_all[:, i])
    auc_i = auc(fpr_i, tpr_i)
    auc_scores[all_class_names[i]] = auc_i
    ax.plot(fpr_i, tpr_i, color=colors[i], lw=2,
            label=f"{all_class_names[i]} (AUC={auc_i:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"Per-Class ROC (OvR) — {best_method_name}")
ax.legend(loc="lower right", fontsize=8)
ax.grid(True, alpha=0.3)

# Macro-average ROC
ax = axes[1]
all_fpr = np.linspace(0, 1, 200)
mean_tpr = np.zeros_like(all_fpr)
valid = 0
for i in range(num_all):
    if y_true_bin[:, i].sum() == 0:
        continue
    fpr_i, tpr_i, _ = roc_curve(y_true_bin[:, i], probs_all[:, i])
    mean_tpr += np.interp(all_fpr, fpr_i, tpr_i)
    valid += 1
mean_tpr /= max(valid, 1)
macro_auc = auc(all_fpr, mean_tpr)
ax.plot(all_fpr, mean_tpr, "navy", lw=2, label=f"Macro-Avg (AUC={macro_auc:.3f})")
ax.fill_between(all_fpr, 0, mean_tpr, alpha=0.1, color="navy")
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Macro-Average ROC")
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "roc_curves.png"), dpi=150)
plt.show()

# ── Precision-Recall Curves ──
fig, ax = plt.subplots(figsize=(10, 6))
ap_scores = {}
for i in range(num_all):
    if y_true_bin[:, i].sum() == 0:
        continue
    prec_i, rec_i, _ = precision_recall_curve(y_true_bin[:, i], probs_all[:, i])
    ap_i = average_precision_score(y_true_bin[:, i], probs_all[:, i])
    ap_scores[all_class_names[i]] = ap_i
    ax.plot(rec_i, prec_i, color=colors[i], lw=2,
            label=f"{all_class_names[i]} (AP={ap_i:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall Curves — {best_method_name}")
ax.legend(loc="lower left", fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "pr_curves.png"), dpi=150)
plt.show()

# Print scores
print("AUC per class:")
for c, a in auc_scores.items():
    print(f"  {c}: {a:.4f}")
print(f"  Macro-Avg: {macro_auc:.4f}")
print("\nAverage Precision per class:")
for c, a in ap_scores.items():
    print(f"  {c}: {a:.4f}")
print(f"  mAP: {np.mean(list(ap_scores.values())):.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 23: Confusion Matrix
# ═══════════════════════════════════════════════════════════════

cm = confusion_matrix(y_true_all, y_pred_all, labels=list(range(num_all)))

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_title(f"Confusion Matrix — {best_method_name}", fontsize=14)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ticks = np.arange(num_all)
ax.set_xticks(ticks)
ax.set_xticklabels(all_class_names, rotation=45, ha="right")
ax.set_yticks(ticks)
ax.set_yticklabels(all_class_names)

# Annotate cells with counts
thresh = cm.max() / 2.0
for i in range(num_all):
    for j in range(num_all):
        color = "white" if cm[i, j] > thresh else "black"
        ax.text(j, i, f"{cm[i, j]:,}",
                ha="center", va="center", color=color, fontsize=9)

fig.colorbar(im)
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=150)
plt.show()
print(f"Saved: {os.path.join(SAVE_DIR, 'confusion_matrix.png')}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 24: Open-Set Metrics Summary
# ═══════════════════════════════════════════════════════════════

best_metrics = compute_method_metrics(best_pk, best_pu, y_test_known, NUM_KNOWN)

print("=" * 60)
print("  OPEN-SET RECOGNITION METRICS SUMMARY")
print(f"  Best Method: {best_method_name}")
print("=" * 60)
print(f"  CNN Classifier Accuracy (no OpenMax):  {cls_acc:.4f}")
print(f"  Known Classification Accuracy:         {best_metrics['known_acc']:.4f}")
print(f"  Known Rejection Rate:                  {best_metrics['known_rejection']:.4f}")
print(f"  Unknown Detection Rate:                {best_metrics['unknown_det']:.4f}")
print(f"  False Known Rate:                      {best_metrics['false_known']:.4f}")
print(f"  H-Score (KCA & UDR harmonic mean):     {best_metrics['h_score']:.4f}")
print(f"  Overall Accuracy:                      {best_metrics['overall_accuracy']:.4f}")
print(f"  Macro AUC:                             {macro_auc:.4f}")
print(f"  mAP:                                   {np.mean(list(ap_scores.values())):.4f}")
print("=" * 60)

# Quick comparison across methods
print("\nH-Score comparison:")
print(f"  M1 (Standard OpenMax): {h_score_m1:.4f}")
print(f"  M2 (Hybrid OpenMax):   {h_score_m2:.4f}")
print(f"  M3 (MSP Threshold):    {h_score_m3:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 25: Save All Artifacts
# ═══════════════════════════════════════════════════════════════

# 1. Save trained model
model.save(os.path.join(SAVE_DIR, "nids_classifier.keras"))

# 2. OpenMax parameters (all 3 methods)
def weibull_to_serializable(wp):
    return {str(k): list(v) if v else None for k, v in wp.items()}

def mavs_to_serializable(m):
    return {str(k): v.tolist() for k, v in m.items()}

openmax_data = {
    "method_1_std_openmax_logit": {
        "mavs": mavs_to_serializable(mavs_logit),
        "weibull_params": weibull_to_serializable(weibull_logit),
        "distance_type": "euclidean",
        "space": "logit",
    },
    "method_2_hybrid_openmax": {
        "mavs": mavs_to_serializable(mavs_pen),
        "weibull_params": weibull_to_serializable(weibull_pen),
        "distance_type": "cosine",
        "space": "penultimate",
    },
    "method_3_msp_threshold": {
        "best_threshold": float(best_th),
    },
    "config": {
        "known_classes": KNOWN_CLASSES,
        "unknown_classes": UNKNOWN_CLASSES,
        "num_known": NUM_KNOWN,
        "tail_size": TAIL_SIZE,
        "alpha_rank": ALPHA_RANK,
    },
}
with open(os.path.join(SAVE_DIR, "openmax_params.json"), "w") as f:
    json.dump(openmax_data, f, indent=2)

# 3. MAVs as numpy
np.savez(
    os.path.join(SAVE_DIR, "openmax_mavs.npz"),
    **{f"logit_class_{k}": v for k, v in mavs_logit.items()},
    **{f"pen_class_{k}": v for k, v in mavs_pen.items()},
)

# 4. AUC/AP scores
auc_ap_df = pd.DataFrame({
    "Class": list(auc_scores.keys()),
    "AUC": list(auc_scores.values()),
    "AP": [ap_scores.get(c, np.nan) for c in auc_scores.keys()],
})
auc_ap_df.to_csv(os.path.join(SAVE_DIR, "auc_ap_scores.csv"), index=False)

# 5. Comprehensive metadata
meta = {
    "model_type": "cnn_openmax_final",
    "approach": "supervised_cnn_with_3_openset_methods",
    "num_known": NUM_KNOWN,
    "known_classes": KNOWN_CLASSES,
    "unknown_classes": UNKNOWN_CLASSES,
    "known_class_names": known_class_names,
    "epochs_trained": len(history["loss"]),
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "tail_size": TAIL_SIZE,
    "alpha_rank": ALPHA_RANK,
    "train_time_min": round(train_time / 60, 2),
    "best_val_accuracy": float(best_val_acc),
    "classifier_test_accuracy": float(cls_acc),
    "best_method": best_method_name,
    "results": {
        "m1_std_openmax": {
            "known_acc": float(known_acc_m1),
            "unknown_det": float(unknown_det_m1),
            "h_score": float(h_score_m1),
        },
        "m2_hybrid_openmax": {
            "known_acc": float(known_acc_m2),
            "unknown_det": float(unknown_det_m2),
            "h_score": float(h_score_m2),
        },
        "m3_msp_threshold": {
            "known_acc": float(known_acc_m3),
            "unknown_det": float(unknown_det_m3),
            "h_score": float(h_score_m3),
            "best_threshold": float(best_th),
        },
    },
    "overall_accuracy": float(accuracy_score(y_true_all, y_pred_all)),
    "macro_auc": float(macro_auc),
    "mAP": float(np.mean(list(ap_scores.values()))),
    "train_samples": int(len(X_train)),
    "val_samples": int(len(X_val)),
    "unknown_samples": int(len(X_test_unknown)),
}
with open(os.path.join(SAVE_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

# Print all saved files
print(f"All artifacts saved to: {SAVE_DIR}")
print("-" * 50)
for fname in sorted(os.listdir(SAVE_DIR)):
    fpath = os.path.join(SAVE_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {fname:40s} ({size_mb:.2f} MB)")

## CNN + OpenMax vs InfoGAN + OpenMax

| Aspect | InfoGAN + OpenMax | CNN + OpenMax (this notebook) |
|--------|-------------------|------------------------------|
| **Training paradigm** | Unsupervised (adversarial GAN) | Supervised (labeled classification) |
| **Training stability** | GAN training is unstable, prone to NaN/mode collapse | Standard backprop, very stable |
| **Class learning** | Must discover classes via mutual information maximization | Directly learns class decision boundaries |
| **Training time** | Hours (200+ epochs of adversarial training) | Minutes (10-30 epochs with early stopping) |
| **Mixed precision** | Manual loss scaling needed, frequent NaN issues | Keras handles loss scaling automatically |
| **Open-set detection** | OpenMax on Q-classifier penultimate layer | 3 methods: Standard OpenMax, Hybrid OpenMax, MSP Threshold |
| **Expected known accuracy** | 60-70% (unsupervised discovery has inherent ceiling) | 90%+ (direct supervision) |
| **Penultimate features** | Shared GAN backbone features (may not be class-discriminative) | Dedicated classifier features (optimized for classification) |
| **Key advantage** | No labels needed; can discover novel structure | Higher accuracy; multiple robust open-set methods |
| **Key disadvantage** | Lower accuracy; GAN instability; single detection method | Requires labeled data; cannot discover novel classes |
| **OpenMax bug risk** | Same exponweib/float16 bugs as v1 | Fixed in openmax_kaggle_v2: weibull_min + float64 |
| **Logit vs feature confusion** | Q classifier has limited logit access | Explicit separation: penultimate (128-dim) vs logits (6-dim) |

**Bottom line:** For CICIDS2017 where labels are available, the supervised CNN + OpenMax approach
is strictly superior in classification accuracy. The InfoGAN approach is valuable in scenarios where
labeled data is scarce or unavailable, but its open-set detection performance is limited by the
quality of unsupervised class discovery.